## Training notebook

In [1]:
import gym
import highway_env

from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO
import tensorflow as tf

%load_ext tensorboard
from datetime import datetime
import os
from ipywidgets import interact, widgets
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, BaseCallback

from typing import Callable

### Select operating system

In [2]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UNIX', 'WINDOWS'])

interactive(children=(Dropdown(description='desired_os', options=('UNIX', 'WINDOWS'), value='UNIX'), Output())…

<function __main__.f(desired_os)>

In [3]:
OUTPUT_PATH = '/Users/fornerispighetti/HighwayDRL/training_output/' if os_id.value == "UNIX" else 'C:/Users/pigo/Desktop/HighwayDRL/training_output/'

### Select environment

In [4]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0','multipleO-dm-env-v0'])

interactive(children=(Dropdown(description='input_env', options=('dm-env-v0', 'acc-dm-env-v0', 'jam-dm-env-v0'…

<function __main__.f(input_env)>

In [5]:
total_timesteps = 3e4
# Create and wrap the environment
env = gym.make(env_id.value)
env.configure({
    "training_total_timesteps": total_timesteps
})
# Create a TensorboardCallback to plot sub-rewards
class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=1):
        super(TensorboardCallback, self).__init__(verbose)
        self.coll_rew = 0
        self.rml_rew = 0
        self.high_speed_rew = 0
        self.current_steps = 0

    def _on_step(self) -> bool:        
        self.rml_rew += self.training_env.get_attr('rml_reward')[0]
        self.high_speed_rew += self.training_env.get_attr('high_speed_reward')[0]
            
        if self.locals["dones"][0]:
            self.coll_rew += -self.training_env.get_attr('collision_reward')[0]
            self.logger.record("eval/coll_rew", self.coll_rew)
            self.logger.record("episode/RML_rew", self.rml_rew / self.current_steps)
            self.logger.record("episode/high_speed_rew", self.high_speed_rew / self.current_steps)
            self.rml_rew = self.high_speed_rew = self.current_steps = 0
        else:
            self.current_steps = self.training_env.get_attr('steps')[0]
        return True

# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=False, render=False)

In [6]:
def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate.
    :return: schedule that computes
      current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining:
        :return: current learning rate
        """
        return progress_remaining * initial_value

    return func

In [7]:
ilr = 3.5e-4
# PPO parameters
model = PPO("MlpPolicy",
        env,
        learning_rate=ilr,
        verbose=1,
        tensorboard_log=f'{OUTPUT_PATH}logdir'
        )

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [8]:
# Train the agent for n steps
model.learn(total_timesteps=int(total_timesteps), callback=[checkpoint_callback, eval_callback, TensorboardCallback()])

# Save the trained agent
model.save(f'{OUTPUT_PATH}models/ppo_RL_allPOS_lessRML_{str(os_id.value)}_{total_timesteps:.1E}_{env_id.value}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

Logging to C:/Users/pigo/Desktop/HighwayDRL/training_output/logdir\PPO_5


c:\Users\pigo\miniconda3\envs\highway\lib\site-packages\stable_baselines3\common\evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=1000, episode_reward=38.27 +/- 39.77
Episode length: 43.20 +/- 39.83
---------------------------------
| episode/           |          |
|    RML_rew         | 0.07     |
|    high_speed_rew  | 0.0448   |
| eval/              |          |
|    coll_rew        | 28       |
|    mean_ep_length  | 43.2     |
|    mean_reward     | 38.3     |
| time/              |          |
|    total_timesteps | 1000     |
---------------------------------
New best mean reward!
Eval num_timesteps=2000, episode_reward=41.45 +/- 41.90
Episode length: 45.40 +/- 42.39
---------------------------------
| episode/           |          |
|    RML_rew         | 0.0176   |
|    high_speed_rew  | 0.331    |
| eval/              |          |
|    coll_rew        | 51       |
|    mean_ep_length  | 45.4     |
|    mean_reward     | 41.5     |
| time/              |          |
|    total_timesteps | 2000     |
---------------------------------
New best mean reward!
--------------------------------

### Continued learning

In [9]:
env_cl_id = 'multipleO-dm-env-v0'
env_cl = gym.make(env_cl_id)

In [10]:
custom_objects = { 'learning_rate': 3.5e-4 }

model_cl = PPO.load(f"{OUTPUT_PATH}models/ppo_RL_allPOS_lessRML_WINDOWS_3.0E+04_multipleO-dm-env-v0_12-07_16-23-49", env=env_cl, custom_objects=custom_objects, tensorboard_log=f"{OUTPUT_PATH}logdir")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [11]:
class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=1):
        super(TensorboardCallback, self).__init__(verbose)
        self.coll_rew = 0
        self.rml_rew = 0
        self.high_speed_rew = 0
        self.current_steps = 0

    def _on_step(self) -> bool:        
        self.rml_rew += self.training_env.get_attr('rml_reward')[0]
        self.high_speed_rew += self.training_env.get_attr('high_speed_reward')[0]
            
        if self.locals["dones"][0]:
            self.coll_rew += -self.training_env.get_attr('collision_reward')[0]
            self.logger.record("eval/coll_rew", self.coll_rew)
            self.logger.record("episode/RML_rew", self.rml_rew / self.current_steps)
            self.logger.record("episode/high_speed_rew", self.high_speed_rew / self.current_steps)
            self.rml_rew = self.high_speed_rew = self.current_steps = 0
        else:
            self.current_steps = self.training_env.get_attr('steps')[0]
        return True

# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=False, render=False)

In [12]:
new_timesteps = 3e4

model_cl.learn(total_timesteps=int(new_timesteps), reset_num_timesteps=False, callback=[checkpoint_callback, eval_callback, TensorboardCallback()])

model_cl.save(f'{OUTPUT_PATH}models/ppo_RL_allPOS_lessRML_CL_{str(os_id.value)}_{(total_timesteps + new_timesteps):.1E}_{env_cl_id}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

Logging to C:/Users/pigo/Desktop/HighwayDRL/training_output/logdir\PPO_5
Eval num_timesteps=31720, episode_reward=90.36 +/- 40.00
Episode length: 99.00 +/- 42.00
---------------------------------
| episode/           |          |
|    RML_rew         | 0.0521   |
|    high_speed_rew  | 0.392    |
| eval/              |          |
|    coll_rew        | 0        |
|    mean_ep_length  | 99       |
|    mean_reward     | 90.4     |
| time/              |          |
|    total_timesteps | 31720    |
---------------------------------
New best mean reward!
Eval num_timesteps=32720, episode_reward=115.06 +/- 1.22
Episode length: 120.00 +/- 0.00
---------------------------------
| episode/           |          |
|    RML_rew         | 0.0193   |
|    high_speed_rew  | 0.37     |
| eval/              |          |
|    coll_rew        | 0        |
|    mean_ep_length  | 120      |
|    mean_reward     | 115      |
| time/              |          |
|    total_timesteps | 32720    |
-------------

In [ ]:
model_cl.save(f'{OUTPUT_PATH}models/RECppo_RL_{str(os_id.value)}_1e5_DMenv_cl3_{datetime.datetime.today().day}-{datetime.datetime.today().month}-{datetime.datetime.today().year}')